# **BioPAX-based Pathway Network Projection and Flattening**

This notebook produces the AT_MN family of artifacts 

- BioPax: Biological Pathway Exchange

# Import Libraries and Configurations

In [1]:
import networkx as nx
import pandas as pd
from collections import defaultdict
from itertools import count
from pathlib import Path
from pybiopax import model_from_owl_file
from pybiopax.biopax import (
    Complex,
    Pathway,
    Protein,
)

HSA_FILE = 'Homo-sapiens_v96.owl'
HSA_FILE_PATH = f'../data/external/reactome/{HSA_FILE}'
NETWORKS_DATA_PATH = '..data/processed/networks'

# Dictionary to index pathway objects by Reactome ID
PATHWAYS = dict()

# Counter for pathway locations ID creation
LOCATIONS_COUNTER = count(1)

# Dynamic programming caches
INTERACTION_CACHE = dict()
PROTEIN_COLLECTION_CACHE = dict()
LOCATIONS_CACHE = dict()

# Functions

## BioPAX Pathway DAG

In [2]:
def collect_proteins(entity):
    """
    Recursively collect all protein objects contained within a physical entity.

    This function traverses a physical entity hierarchy and returns all
    :class:`Protein` instances contained directly or indirectly within the
    input entity. It supports individual proteins, protein entity sets
    (via ``member_physical_entity``), and :class:`Complex` objects.
    Results are memoized using a global cache to avoid redundant traversals
    of previously visited entities.

    Parameters
    ----------
    entity : Protein or Complex or None
        The physical entity from which to collect proteins. The entity may be
        a single :class:`Protein`, a protein entity set, a
        :class:`Complex`, or ``None``.

    Returns
    -------
    set of Protein
        A set containing all unique :class:`Protein` instances found within
        the entity. Returns an empty set if ``entity`` is ``None``.

    Notes
    -----
    Protein entity sets are recursively expanded through their
    ``member_physical_entity`` attribute, whereas complexes are traversed
    through their ``component`` attribute. Cache entries are keyed by the
    entity's ``uid`` attribute when available; otherwise, Python's built-in
    :func:`id` is used. The cache is stored in the global
    ``PROTEIN_COLLECTION_CACHE`` dictionary.

    Examples
    --------
    >>> proteins = collect_proteins(complex_entity)
    >>> len(proteins)
    5
    """
    # None object
    if entity is None:
        return set()

    # Get entity object ID
    entity_id = getattr(entity, 'uid', id(entity))

    # Check cache
    if entity_id in PROTEIN_COLLECTION_CACHE:
        return PROTEIN_COLLECTION_CACHE[entity_id]

    # Create a set to store Protein objects
    proteins = set()

    # Get EntitySet members, if present
    member_entities = getattr(
        entity, 'member_physical_entity', None
    )

    # Protein object
    if isinstance(entity, Protein):
        if member_entities:
            for member in member_entities:
                proteins.update(
                    collect_proteins(member)
                )
        else:
            proteins.add(entity)

    # Complex object
    if isinstance(entity, Complex):
        for component in entity.component:
            proteins.update(
                collect_proteins(component)
            )

    # Cache result
    PROTEIN_COLLECTION_CACHE[entity_id] = proteins

    return proteins

In [3]:
def get_interaction_proteins(interaction):
    """
    Extract all proteins participating in an interaction.

    This function collects all unique :class:`Protein` instances associated
    with an interaction by traversing its ``controller``, ``left``, and
    ``right`` participants. Protein extraction is performed recursively using
    :func:`collect_proteins`, allowing proteins nested within complexes or
    entity sets to be included. Results are memoized to avoid repeated
    traversal of previously processed interactions.

    Parameters
    ----------
    interaction : object
        An interaction object exposing one or more of the attributes
        ``controller``, ``left``, and ``right``. Each attribute is expected
        to contain an iterable of physical entities.

    Returns
    -------
    set of Protein
        A set containing all unique proteins participating in the interaction.

    Notes
    -----
    Cache entries are keyed by the interaction's ``uid`` attribute when
    available; otherwise, Python's built-in :func:`id` is used. The cache is
    stored in the global ``INTERACTION_CACHE`` dictionary.

    Examples
    --------
    >>> proteins = get_interaction_proteins(interaction)
    >>> len(proteins)
    12
    """
    # Get interaction object ID
    interaction_id = getattr(
        interaction, 'uid', id(interaction)
    )

    # Check if the interaction has been visited previously
    if interaction_id in INTERACTION_CACHE:
        return INTERACTION_CACHE[interaction_id]

    # Create a set to store Protein objects
    proteins = set()

    # Iterate over the attributes of interest
    for attr in ('controller', 'left', 'right'):
        # Get the object(s) contained in the attribute
        entities = getattr(interaction, attr, None)

        # Attribute with an empty object
        if not entities:
            continue
        
        # Iterate over entity(ies) to extract its proteins
        for entity in entities:
            proteins.update(
                collect_proteins(entity)
            )

    # Add interaction and its proteins to the cache
    INTERACTION_CACHE[interaction_id] = proteins

    return proteins

In [4]:
def build_pathway_dag(G, pathway, parent_id, visited=None):
    """
    Build a directed acyclic graph (DAG) representation of a pathway hierarchy.

    This function recursively traverses a pathway and populates a directed
    graph with pathway steps, interactions, nested pathways, and their
    associated proteins. The resulting graph captures the hierarchical
    structure of the pathway and the relationships between pathway steps,
    biological processes, and participating proteins.

    Nodes are annotated with a human-readable name and a ``type``
    attribute indicating their biological type (e.g., ``PathwayStep``,
    ``Protein``, or the interaction class name). Edges represent containment
    or participation relationships.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed graph to populate. The graph is modified in place.
    pathway : Pathway
        Root pathway to traverse.
    parent_id : str
        Identifier of the parent node to which pathway steps will be
        connected.
    visited : set, optional
        Set of pathway identifiers that have already been traversed. This is
        used internally to prevent infinite recursion when pathways reference
        one another. If ``None``, an empty set is created.

    Returns
    -------
    networkx.DiGraph
        The populated directed graph.

    Notes
    -----
    The traversal proceeds as follows:

    1. Add pathway steps as child nodes of ``parent_id``.
    2. Add each process associated with a pathway step.
    3. Recursively traverse nested pathways.
    4. For interaction processes, add all participating proteins obtained via
       :func:`get_interaction_proteins`.

    Existing nodes are reused rather than recreated, allowing multiple
    relationships to converge on the same biological entity.

    Examples
    --------
    >>> import networkx as nx
    >>> G = nx.DiGraph()
    >>> build_pathway_dag(
    ...     G,
    ...     pathway=root_pathway,
    ...     parent_id=root_pathway.uid
    ... )
    >>> G.number_of_nodes()
    153
    """
    # Check if the set of visited pathways has been initialized
    if visited is None:
        visited = set()

    # Get pathway object ID
    pathway_id = getattr(pathway, 'uid', id(pathway))

    # Check if the pathway has already been visited
    if pathway_id in visited:
        return G

    # Add the pathway to the set of visited pathways
    visited.add(pathway_id)

    # Get the order of the pathway components
    pathway_order = getattr(pathway, 'pathway_order', None)
    
    if not pathway_order:
        return G

    # Iterate over the pathway step(s)
    for step in pathway_order:
        # Add pathway step node to the graph, if necessary
        if step.uid not in G:
            G.add_node(
                step.uid,
                name=getattr(
                    step, 'display_name', step.uid
                ),
                type='PathwayStep'
            )

        # Add edge from pathway to pathway step
        G.add_edge(parent_id, step.uid)

        # Iterate over the process(es) in the pathway step
        for process in step.step_process:
            # Add process node to the graph, if necessary
            if process.uid not in G:
                process_name = (
                    getattr(process, 'display_name', None)
                    or process.uid
                )

                if process_name == process.uid:
                    process_name = (
                        getattr(process, 'control_type', None)
                        or process.uid
                    )

                node_attributes = {
                    'name': process_name,
                    'type': type(process).__name__
                }

                # Store controlled attribute for Control instances
                if hasattr(process, 'controlled'):
                    controlled = getattr(process, 'controlled', None)

                    if controlled is not None:
                        node_attributes['controlled'] = (
                            controlled.uid
                            if hasattr(controlled, 'uid')
                            else controlled
                        )

                G.add_node(
                    process.uid,
                    **node_attributes
                )

            # Add edge from pathway step to process
            G.add_edge(step.uid, process.uid)

            # Nested Pathway object
            if isinstance(process, Pathway):
                # Traverse the subpathway
                build_pathway_dag(
                    G=G,
                    pathway=process,
                    parent_id=process.uid,
                    visited=visited
                )
                continue

            # Get proteins related to the Interaction object
            proteins = get_interaction_proteins(process)

            # Iterate over the proteins
            for protein in proteins:
                # Add protein node to the graph, if necessary
                if protein.uid not in G:
                    protein_name = (
                        getattr(protein, 'display_name', None)
                        or protein.uid
                    )

                    G.add_node(
                        protein.uid,
                        name=protein_name,
                        type='Protein'
                    )

                # Add edge from process to protein
                G.add_edge(process.uid, protein.uid)

    return G

## Tripartite PSP Network

In [5]:
def merge_control_reaction_nodes(G):
    """
    Merge Control nodes with their controlled biochemical reactions.

    A merge is performed only if:
      1. The Control node references an existing Reaction node via
         the 'controlled' attribute.
      2. The Control and Reaction have the exact same external predecessors.

    The merged node:
      - has a deterministic ID: merge_<control_id>_<reaction_id>
      - has a merged name
      - has a merged type
      - preserves the control_type attribute
      - inherits all incoming and outgoing edges
      - preserves edge attributes
      - avoids duplicate edges

    Parameters
    ----------
    G : nx.DiGraph
        Graph containing Control nodes with a 'controlled' attribute.

    Returns
    -------
    nx.DiGraph
        A modified copy of the original graph.
    """
    # Preserve the original graph
    G = G.copy()

    control_nodes = [
        node_id
        for node_id, data in G.nodes(data=True)
        if 'controlled' in data
    ]

    for control_id in control_nodes:

        if control_id not in G:
            continue

        control_data = G.nodes[control_id]
        reaction_id = control_data['controlled']

        if reaction_id not in G:
            continue

        reaction_data = G.nodes[reaction_id]

        # --------------------------------------------------------------
        # Verify identical external predecessors
        # --------------------------------------------------------------
        control_preds = set(G.predecessors(control_id))
        reaction_preds = set(G.predecessors(reaction_id))

        # Ignore the direct Control → Reaction edge
        control_preds.discard(reaction_id)
        reaction_preds.discard(control_id)

        if control_preds != reaction_preds:
            continue

        # --------------------------------------------------------------
        # Create merged node
        # --------------------------------------------------------------
        merged_id = f'merge_{control_id}_{reaction_id}'

        merged_name = (
            f'{reaction_data.get('name', reaction_id)} '
            f'[{control_data.get('name', control_id)}]'
        )

        control_type = control_data.get('type', 'Control')
        reaction_type = reaction_data.get('type', 'Reaction')
        merged_type = f'{control_type}_{reaction_type}'

        G.add_node(
            merged_id,
            name=merged_name,
            type=merged_type,
            control_type=control_data.get('control_type'),
        )

        # --------------------------------------------------------------
        # Collect inherited edges (deduplicated)
        # --------------------------------------------------------------
        edges = {}

        def add_edge(u, v, attrs):
            if (u, v) not in edges:
                edges[(u, v)] = dict(attrs)

        # Incoming to Control
        for source, _, attrs in G.in_edges(control_id, data=True):
            if source not in (control_id, reaction_id):
                add_edge(source, merged_id, attrs)

        # Outgoing from Control
        for _, target, attrs in G.out_edges(control_id, data=True):
            if target not in (control_id, reaction_id):
                add_edge(merged_id, target, attrs)

        # Incoming to Reaction
        for source, _, attrs in G.in_edges(reaction_id, data=True):
            if source not in (control_id, reaction_id):
                add_edge(source, merged_id, attrs)

        # Outgoing from Reaction
        for _, target, attrs in G.out_edges(reaction_id, data=True):
            if target not in (control_id, reaction_id):
                add_edge(merged_id, target, attrs)

        # --------------------------------------------------------------
        # Remove original nodes
        # --------------------------------------------------------------
        G.remove_nodes_from([control_id, reaction_id])

        # --------------------------------------------------------------
        # Restore inherited edges
        # --------------------------------------------------------------
        for (u, v), attrs in edges.items():
            if not G.has_edge(u, v):
                G.add_edge(u, v, **attrs)

    return G

In [6]:
def project_pathway_steps(G):
    """
    Remove pathway step nodes from a pathway graph while preserving connectivity.

    This function projects a pathway graph onto its non-step nodes by removing
    all nodes whose ``type`` is ``"PathwayStep"``. Before each pathway
    step is removed, directed edges are added from each predecessor of the step
    to each of its successors, preserving reachability in the resulting graph.

    The input graph is copied before modification, leaving the original graph
    unchanged.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed pathway graph containing nodes annotated with a
        ``type`` attribute.

    Returns
    -------
    networkx.DiGraph
        A copy of the input graph with all ``PathwayStep`` nodes removed and
        replacement edges added to preserve connectivity.

    Notes
    -----
    For each pathway step node, the function creates the Cartesian product of
    its predecessors and successors, adding an edge for every predecessor–
    successor pair before removing the step node.

    Examples
    --------
    >>> G_projected = project_pathway_steps(G)
    >>> any(
    ...     attr['type'] == 'PathwayStep'
    ...     for _, attr in G_projected.nodes(data=True)
    ... )
    False
    """
    # Create a copy of the graph
    G_proj = G.copy()

    # Get the pathway step nodes
    step_nodes = [
        n for n, attr in G_proj.nodes(data=True)
        if attr['type'] == 'PathwayStep'
    ]

    # Iterate over each pathway step node
    for step in step_nodes:
        # Get node parent(s) and child(ren)
        parents = tuple(G_proj.predecessors(step))
        children = tuple(G_proj.successors(step))

        # Add edges between parent(s) and child(ren)
        G_proj.add_edges_from(
            (p, c)
            for p in parents
            for c in children
        )

        # Remove the node from the graph
        G_proj.remove_node(step)

    return G_proj

In [7]:
def identify_pp_paths(G, source):
    """
    Enumerate all pathway-to-protein (PP) paths from a source node.

    This function traverses a directed acyclic graph (DAG) in topological
    order and yields every directed path from the specified source node to
    each reachable protein node. Protein nodes are identified by their
    ``type`` attribute equal to ``"Protein"``.

    Paths are propagated incrementally through the graph, allowing every
    distinct route from the source to a protein to be enumerated without
    performing repeated graph traversals.

    Parameters
    ----------
    G : networkx.DiGraph
        Directed acyclic graph containing nodes annotated with a
        ``type`` attribute.
    source : hashable
        Identifier of the source node from which pathway-to-protein paths are
        generated.

    Yields
    ------
    list
        A list of node identifiers representing a directed pathway-to-protein
        (PP) path from ``source`` to a protein node. Each yielded path begins
        with ``source`` and ends with a node whose ``type`` is
        ``"Protein"``.

    Notes
    -----
    This function assumes that ``G`` is a directed acyclic graph (DAG). The
    traversal relies on :func:`networkx.topological_sort` and propagates
    partial paths through successor nodes until protein leaves are reached.

    Examples
    --------
    >>> for path in identify_pp_paths(G, source='R-HSA-12345'):
    ...     print(path)
    ['R-HSA-12345', 'reaction1', 'P12345']
    ['R-HSA-12345', 'reaction2', 'complex1', 'P67890']
    """
    # Cache useful objects
    type = nx.get_node_attributes(G, 'type')
    successors = G.successors

    # Create a dictionary of empty lists to store paths
    paths = defaultdict(list)

    # Initialize the list of paths from the source node
    paths[source].append([source])

    # Iterate over the nodes in topological order
    for node in nx.topological_sort(G):
        # Get all paths that lead to the node
        current_paths = paths.get(node)

        # Skip unreachable node
        if not current_paths:
            continue

        # Output protein (leaf) paths
        if type[node] == 'Protein':
            yield from current_paths
            continue

        # Iterate over the node's child(ren)
        for child in successors(node):
            # Cache the list of paths from the child node
            child_paths = paths[child]

            # Iterate over the paths the lead to the node
            for path in current_paths:
                # Propagate paths to child
                child_paths.append(path + [child])

In [8]:
def export_network_tables(G):
    """
    Export a NetworkX graph as edge and node tables.

    This function converts a directed graph into two pandas DataFrames: one
    containing the graph edges and another containing the node attributes.

    Parameters
    ----------
    G : networkx.DiGraph
        Graph to export.

    Returns
    -------
    tuple of pandas.DataFrame
        A tuple ``(edges_df, nodes_df)`` where:

        - ``edges_df`` contains one row per directed edge with the default
        ``source`` and ``target`` columns produced by
        :func:`networkx.to_pandas_edgelist`.
        - ``nodes_df`` contains one row per node with an ``id`` column
        followed by all node attributes stored in the graph.

    Notes
    -----
    The edge table is generated using
    :func:`networkx.to_pandas_edgelist`, preserving the default
    ``source`` and ``target`` column names. The node table is
    constructed from the graph nodes and their associated attributes.

    Examples
    --------
    >>> edges_df, nodes_df = export_network_tables(G)
    >>> edges_df.head()
    >>> nodes_df.head()
    """
    # Create a DataFrame for the edge list
    edges_df = nx.to_pandas_edgelist(G)

    # Create a DataFrame for the node attributes
    nodes_df = pd.DataFrame(
        (
            {'id': n, **attrs}
            for n, attrs in G.nodes(data=True)
        )
    )

    return edges_df, nodes_df

In [9]:
def create_psp_network(pathway_id):
    """
    Create a pathway-semantics-protein (PSP) network from a pathway.

    This function constructs a simplified tripartite representation of a
    biological pathway by connecting the pathway to semantic descriptions of
    pathway-to-protein paths and, subsequently, to the associated protein nodes.
    The resulting network contains three node types:

    - ``Pathway``: the root pathway.
    - ``Pathway Location``: textual representations of the biological process 
    sequence connecting a pathway to a protein.
    - ``Messenger RNA``: protein nodes identified from pathway interactions.

    The PSP network is generated through the following steps:

    1. Retrieve the pathway object from the global ``PATHWAYS`` collection.
    2. Construct the complete pathway DAG using :func:`build_pathway_dag`.
    3. Remove intermediate ``PathwayStep`` nodes using
    :func:`project_pathway_steps`.
    4. Enumerate all pathway-to-protein (PP) paths using
    :func:`identify_pp_paths`.
    5. Convert each PP path into a semantic representation by joining
    intermediate process names.
    6. Construct a tripartite network containing
    ``Pathway → Pathway Location → Messenger RNA`` relationships.
    7. Export the resulting graph as node and edge tables.

    Parameters
    ----------
    pathway_id : str
        Identifier of the pathway stored in the global ``PATHWAYS`` collection.

    Returns
    -------
    tuple of pandas.DataFrame
        A tuple ``(edges_df, nodes_df)`` containing the edge and node tables
        representing the PSP network, as produced by
        :func:`export_network_tables`.

    Raises
    ------
    KeyError
        If ``pathway_id`` is not present in the global ``PATHWAYS`` collection.

    Notes
    -----
    Semantic nodes are generated by concatenating the names of intermediate
    nodes along pathway-to-protein paths using ``" > "`` as a separator.
    Different PP paths that produce identical semantic strings are represented
    by the same semantic node.

    The ``Messenger RNA`` node type corresponds to proteins collected from the
    pathway graph. The naming follows the PSP network convention used for
    downstream analysis.

    Examples
    --------
    >>> edges_df, nodes_df = create_psp_network('R-HSA-123456')
    >>> nodes_df['type'].value_counts()
    """
    # Get the pathway object
    if pathway_id not in PATHWAYS:
        raise KeyError(
            f'Pathway ID {pathway_id!r} not found in PATHWAYS.'
        )

    pathway_obj = PATHWAYS[pathway_id]
    
    # Create an object to represent the pathway DAG
    G = nx.DiGraph()

    # Add the pathway node (root) to the graph
    G.add_node(
        pathway_obj.uid,
        name=pathway_obj.display_name,
        type='Pathway',
    )

    # Traverse the pathway
    build_pathway_dag(
        G=G,
        pathway=pathway_obj,
        parent_id=pathway_obj.uid,
    )

    # Merge nodes
    G_merged = merge_control_reaction_nodes(G)

    # Project the pathway step nodes
    G_proj = project_pathway_steps(G_merged)

    # Create an object to represent the PSP graph
    T = nx.DiGraph()

    # Add the pathway node to the tripartite graph
    nodes = G_proj.nodes
    pathway_name = nodes[pathway_obj.uid]['name']
    T.add_node(
        pathway_id,
        label=pathway_name,
        type='Pathway'
    )

    # Iterate over each pathway-protein (PP) path
    for path in identify_pp_paths(
        G=G_proj, source=pathway_obj.uid,
    ):
        # Get protein name
        protein = nodes[path[-1]]['name']

        # Add protein node to the graph, if necessary
        if protein not in T:
            T.add_node(
                protein,
                label=protein,
                type='Messenger RNA'
            )
        
        # Create PP path textual representation
        semantics = ' > '.join(
            nodes[n]['name']
            for n in path[1:-1]
        )

        # Retrieve or create semantics ID
        if semantics not in LOCATIONS_CACHE:
            # Create semantics ID
            semantics_id = (
                f'S{next(LOCATIONS_COUNTER):05d}'
            )

            # Cache semantics ID
            LOCATIONS_CACHE[semantics] = semantics_id
        else:
            # Retrieve the semantics ID
            semantics_id = LOCATIONS_CACHE[semantics]

        # Add semnatics node to the graph, if necessary
        if semantics_id not in T:
            T.add_node(
                semantics_id,
                label=semantics,
                type='Pathway Location'
            )

        # Add edge from pathway to semantics
        T.add_edge(pathway_id, semantics_id)

        # Add edge from semantics to protein
        T.add_edge(semantics_id, protein)

    return export_network_tables(T)

## Layered Quadripartite Network

In [10]:
def create_network_dfs(group, network):
    base_data_path = '../data/processed/networks/'
    group_data_path = f'{base_data_path}/{group}'

    network_group_file_path = f'{group_data_path}/{network}'

    # Create a DataFrame for the edge list
    edges_file_path = f'{network_group_file_path}-edges.csv'
    edges_df = pd.read_csv(edges_file_path)

    # Create a DataFrame for the node attributes
    nodes_file_path = f'{network_group_file_path}-nodes.csv'
    nodes_df = pd.read_csv(nodes_file_path)

    return edges_df, nodes_df

In [11]:
def create_msp_network(psp_edges_df, psp_nodes_df, mrna_ids):
    # Swap `source` and `target` column names
    psp_edges_df = psp_edges_df \
        .rename(columns={'source': '__temp__'})
    psp_edges_df = psp_edges_df \
        .rename(columns={'target': 'source'})
    psp_edges_df = psp_edges_df \
        .rename(columns={'__temp__': 'target'})
    psp_edges_df = psp_edges_df[['source', 'target']]

    # Get the proteins (mRNAs) of interest
    protein_ids = psp_nodes_df \
        .query('id in @mrna_ids') \
        ['id'].to_list()

    # Get the semantics of interest
    semantics_ids = psp_edges_df \
        .query('source in @protein_ids') \
        ['target'].to_list()

    # Get the pathways of interest
    pathway_ids = psp_edges_df \
        .query('source in @semantics_ids') \
        ['target'].to_list()

    # Combine the nodes of the subgraph in a set
    selected_nodes = (
        set(protein_ids) 
        | set(semantics_ids) 
        | set(pathway_ids)
    )

    # Filter the subgraph edge list
    msp_edges_df = psp_edges_df \
        .query(
            '(source in @selected_nodes)'
            ' and (target in @selected_nodes)'
        ) \
        .reset_index(drop=True)
    
    # Filter the subgraph node attributes
    msp_nodes_df = psp_nodes_df \
        .query('id in @selected_nodes') \
        .reset_index(drop=True)

    return msp_edges_df, msp_nodes_df

In [12]:
def combine_msp_and_mm_networks(
    msp_edges_df, msp_nodes_df, group
):
    # Interaction network (miR-mRNA)
    mm_edges_df, mm_nodes_df = \
        create_network_dfs(
            group, network='interaction-network'
        )

    # Get the expressed mRNAs associated with semantics
    mrna_ids = msp_nodes_df \
        .query('type == "Messenger RNA"') \
        ['id'].to_list()

    # Get the miRs related to the mRNAs of interest
    mir_ids = mm_edges_df \
        .query('target in @mrna_ids') \
        ['source'].to_list()

    # Combine the miR and mRNA nodes in a set
    selected_nodes = (
        set(mir_ids) | set(mrna_ids)
    )

    # Filter the miR and mRNA edge list
    mp_edges_df = mm_edges_df \
        .query(
            '(source in @selected_nodes)'
            ' and (target in @selected_nodes)'
        ) \
        .reset_index(drop=True)

    # Filter the miR and mRNA node attributes
    mp_nodes_df = mm_nodes_df \
        .query('id in @selected_nodes') \
        .reset_index(drop=True)

    # Concatenate edge lists
    lqn_edges_df = pd.concat(
        [mp_edges_df, msp_edges_df],
        ignore_index=True
    ) \
    [['source', 'target']] \
    .drop_duplicates()

    # Concatenate node attributes
    lqn_nodes_df = pd.concat(
        [mp_nodes_df, msp_nodes_df],
        ignore_index=True
    ) \
    .drop_duplicates()

    return lqn_edges_df, lqn_nodes_df

In [13]:
def create_lqn_network(group):
    # Membership network (mRNA-pathway)
    mp_edges_df, mp_nodes_df = \
        create_network_dfs(
            group, network='membership-network'
        )

    # Remove `REAC:` prefix from columns
    mp_edges_df['target'] = \
        mp_edges_df['target'].str.replace('REAC:', '')
    mp_nodes_df['id'] = \
        mp_nodes_df['id'].str.replace('REAC:', '')

    # Get the pathways contained in the membership network
    pathway_ids = mp_nodes_df \
        .query('type == "Pathway"')['id'].to_list()
    
    # Global DataFrames
    group_edges_df = pd.DataFrame(columns=['source', 'target'])
    group_nodes_df = pd.DataFrame(columns=['id', 'label', 'type'])

    # Iterate over the pathways in the membership network
    for pathway_id in pathway_ids:
        # Get the mRNAs (proteins) associated to the pathway
        expressed_mrnas = mp_edges_df \
            .query(f'target == "{pathway_id}"') \
            ['source'].to_list()
        
        # Create the pathway-semantics-protein network
        psp_edges_df, psp_nodes_df = create_psp_network(pathway_id)

        # mRNA-semantics-pathway network
        msp_edges_df, msp_nodes_df = create_msp_network(
            psp_edges_df, psp_nodes_df, expressed_mrnas
        )

        # Concatenate the pathway edge list with the others
        group_edges_df = pd.concat(
            [group_edges_df, msp_edges_df],
            ignore_index=True
        )

        # Concatenate the pathway node attributes with the others
        group_nodes_df = pd.concat(
            [group_nodes_df, msp_nodes_df],
            ignore_index=True
        )

    # Remove duplicates
    group_edges_df = group_edges_df \
        .drop_duplicates(ignore_index=True)
    group_nodes_df = group_nodes_df \
        .drop_duplicates(ignore_index=True)
    
    # Combine mRNA-semantics-pathway with miR-mRNA network
    lqn_edges_df, lqn_nodes_df = \
        combine_msp_and_mm_networks(
            group_edges_df, group_nodes_df, group
        )

    return lqn_edges_df, lqn_nodes_df

# Preparation

## Create the Model

In [14]:
# Create a BioPAX model from the OWL file
model = model_from_owl_file(HSA_FILE_PATH)

Processing OWL elements:   0%|          | 0.00/545k [00:00<?, ?it/s]

## Index the Pathways

In [15]:
# Iterate over the Pathway objects
for pathway in model.get_objects_by_type(Pathway):
    # Iterate over the xrefs of the Pathway object
    for xref in pathway.xref:
        # Add the Reactome Pathway object to the dictionary
        if getattr(xref, 'db', None) == 'Reactome':
            reactome_id = xref.id
            PATHWAYS[reactome_id] = pathway
            break

# Validation Pathways

## Signaling by WNT Pathway

In [16]:
pathway_id = 'R-HSA-195721'

# Create the tripartite pathway-semantics-protein network
edges_df, nodes_df = create_psp_network(pathway_id)

In [17]:
edges_df

,source,target
0,R-HSA-195721,S00001
1,R-HSA-195721,S00002
2,R-HSA-195721,S00003
3,R-HSA-195721,S00004
4,R-HSA-195721,S00005
...,...,...
1918,S00158,CTNNB1
1919,S00158,TCF3
1920,S00158,TCF7L2
1921,S00158,LEF1


In [18]:
nodes_df

,id,label,type
0,R-HSA-195721,Signaling by WNT,Pathway
1,USP34,USP34,Messenger RNA
2,S00001,TCF dependent signaling in response to WNT > U...,Pathway Location
3,APC,APC,Messenger RNA
4,S00002,Degradation of beta-catenin by the destruction...,Pathway Location
...,...,...,...
528,S00158,TCF dependent signaling in response to WNT > F...,Pathway Location
529,TCF3,TCF3,Messenger RNA
530,TCF7L2,TCF7L2,Messenger RNA
531,LEF1,LEF1,Messenger RNA


In [19]:
nodes_df \
    .query('type == "Semantics"')\
    .sort_values(by='label') \
    ['label'].to_list()

[]

## Signaling by Receptor Tyrosine Kinases


In [20]:
pathway_id = 'R-HSA-9006934'

# Create the tripartite pathway-semantics-protein network
edges_df, nodes_df = create_psp_network(pathway_id)

In [21]:
edges_df

,source,target
0,R-HSA-9006934,S00159
1,R-HSA-9006934,S00160
2,R-HSA-9006934,S00161
3,R-HSA-9006934,S00162
4,R-HSA-9006934,S00163
...,...,...
4663,S00964,PKB beta
4664,S00964,PDE3B
4665,S00964,p-S295-PDE3B
4666,S00965,p-S295-PDE3B


In [22]:
nodes_df

,id,label,type
0,R-HSA-9006934,Signaling by Receptor Tyrosine Kinases,Pathway
1,DLG4,DLG4,Messenger RNA
2,S00159,Signaling by ERBB4 > ERBB4 binds DLG4,Pathway Location
3,RON_HUMAN,RON_HUMAN,Messenger RNA
4,S00160,Signaling by MST1 > MST1R autophosphorylates [...,Pathway Location
...,...,...,...
1422,S00964,Signaling by Type 1 Insulin-like Growth Factor...,Pathway Location
1423,PDE3B,PDE3B,Messenger RNA
1424,p-S295-PDE3B,p-S295-PDE3B,Messenger RNA
1425,S00965,Signaling by Insulin receptor > Insulin recept...,Pathway Location


In [23]:
nodes_df \
    .query('type == "Semantics"')\
    .sort_values(by='label') \
    ['label'].to_list()

[]

# Layered Quadripartite Networks

## Basal-like

In [24]:
group = 'basal-like'

basal_edges_df, basal_nodes_df = create_lqn_network(group)

base_data_path = '../data/processed/networks/'
group_file_path = f'{base_data_path}/{group}'

edges_file_path = f'{group_file_path}/lqn-edges.csv'
basal_edges_df.to_csv(edges_file_path, index=False)

nodes_file_path = f'{group_file_path}/lqn-nodes.csv'
basal_nodes_df.to_csv(nodes_file_path, index=False)

In [25]:
# basal_edges_df.to_csv('basal-like_lqn-edges.csv', index=False)
# basal_nodes_df.to_csv('basal-like_lqn-nodes.csv', index=False)

In [26]:
basal_edges_df

,source,target
0,hsa-miR-23a-3p,RXRG
1,S00971,R-HSA-383280
2,S00972,R-HSA-383280
3,RXRG,S00971
4,RXRG,S00972


In [27]:
basal_nodes_df

,id,label,type
0,hsa-miR-23a-3p,hsa-miR-23a-3p,MicroRNA
1,RXRG,RXRG,Messenger RNA
2,R-HSA-383280,Nuclear Receptor transcription pathway,Pathway
3,S00971,Formation of NR-NCOR Corepressor Complex,Pathway Location
4,S00972,Formation of NR-MED1 Coactivator Complex,Pathway Location


### Exploratory Analysis

#### Messenger RNAs Out-Degree

In [28]:
pd.merge(
    left=basal_edges_df,
    right=basal_nodes_df,
    how='inner',
    left_on='source',
    right_on='id'
) \
.query('type == "Messenger RNA"') \
.value_counts(subset=['id', 'label'])

id    label
RXRG  RXRG     2
Name: count, dtype: int64

#### Semantics' In-Degree

In [29]:
pd.merge(
    left=basal_edges_df,
    right=basal_nodes_df,
    how='inner',
    left_on='target',
    right_on='id'
) \
.query('type == "Semantics"') \
.value_counts(subset=['id', 'label'])

Series([], Name: count, dtype: int64)

#### Semantics' Out-Degree

In [30]:
pd.merge(
    left=basal_edges_df,
    right=basal_nodes_df,
    how='inner',
    left_on='source',
    right_on='id'
) \
.query('type == "Semantics"') \
.value_counts(subset=['id', 'label'])

Series([], Name: count, dtype: int64)

#### Pathways In-Degree

In [31]:
pd.merge(
    left=basal_edges_df,
    right=basal_nodes_df,
    how='inner',
    left_on='target',
    right_on='id'
) \
.query('type == "Pathway"') \
.value_counts(subset=['id', 'label'])

id            label                                 
R-HSA-383280  Nuclear Receptor transcription pathway    2
Name: count, dtype: int64

## Luminal A

In [32]:
group = 'luminal-a'

luma_edges_df, luma_nodes_df = create_lqn_network(group)

base_data_path = '../data/processed/networks/'
group_file_path = f'{base_data_path}/{group}'

edges_file_path = f'{group_file_path}/lqn-edges.csv'
luma_edges_df.to_csv(edges_file_path, index=False)

nodes_file_path = f'{group_file_path}/lqn-nodes.csv'
luma_nodes_df.to_csv(nodes_file_path, index=False)

In [33]:
# luma_edges_df.to_csv('luminal-a_lqn-edges.csv', index=False)
# luma_nodes_df.to_csv('luminal-a_lqn-nodes.csv', index=False)

In [34]:
luma_edges_df

,source,target
0,hsa-miR-29c-3p,SERPINH1
1,hsa-miR-193b-3p,KIT
2,hsa-miR-141-3p,SOX17
3,hsa-miR-210-3p,HIF3A
4,hsa-miR-96-5p,ADAM23
...,...,...
338,NRP1,S06426
339,NRP1,S06427
340,NRP1,S06428
341,SRGAP1,S06458


In [35]:
luma_nodes_df

,id,label,type
0,hsa-miR-200a-3p,hsa-miR-200a-3p,MicroRNA
1,hsa-miR-141-3p,hsa-miR-141-3p,MicroRNA
2,hsa-miR-96-5p,hsa-miR-96-5p,MicroRNA
3,hsa-miR-29b-3p,hsa-miR-29b-3p,MicroRNA
4,hsa-miR-342-3p,hsa-miR-342-3p,MicroRNA
...,...,...,...
216,S06426,Axon guidance > Signaling by ROBO receptors > ...,Pathway Location
217,S06427,Axon guidance > L1CAM interactions > Signal tr...,Pathway Location
218,S06428,Axon guidance > L1CAM interactions > CHL1 inte...,Pathway Location
219,S06458,Axon guidance > Signaling by ROBO receptors > ...,Pathway Location


### Exploratory Analysis

#### Messenger RNAs Out-Degree

In [36]:
pd.merge(
    left=luma_edges_df,
    right=luma_nodes_df,
    how='inner',
    left_on='source',
    right_on='id'
) \
.query('type == "Messenger RNA"') \
.value_counts(subset=['id', 'label'])

id        label   
NRP1      NRP1        34
TGFBR2    TGFBR2      29
KIT       KIT         18
PDGFRA    PDGFRA      14
AR        AR          11
RARB      RARB         8
SRGAP1    SRGAP1       7
SERPINH1  SERPINH1     6
NEDD4L    NEDD4L       6
SOX17     SOX17        4
LAMA2     LAMA2        4
MEF2D     MEF2D        4
MSN       MSN          4
ADAM23    ADAM23       3
MYLK      MYLK         3
FOXO1     FOXO1        2
FLI1      FLI1         2
SEMA5A    SEMA5A       1
HIF3A     HIF3A        1
Name: count, dtype: int64

#### Semantics In-Degree

In [37]:
pd.merge(
    left=luma_edges_df,
    right=luma_nodes_df,
    how='inner',
    left_on='target',
    right_on='id'
) \
.query('type == "Semantics"') \
.value_counts(subset=['id', 'label'])

Series([], Name: count, dtype: int64)

In [38]:
luma_edges_df \
    .query(
        '(target == "S02547")'
        + '| (target == "S02548")'
    ) \
    .sort_values(by='target')

,source,target
179,TGFBR2,S02547
180,NEDD4L,S02547
181,TGFBR2,S02548
182,NEDD4L,S02548


In [39]:
luma_edges_df \
    .query(
        '(source == "S02548")'
        + '| (source == "S02549")'
    ) \
    .sort_values(by='source')

,source,target
108,S02548,R-HSA-162582
109,S02549,R-HSA-162582


#### Semantics Out-Degree

In [40]:
pd.merge(
    left=luma_edges_df,
    right=luma_nodes_df,
    how='inner',
    left_on='source',
    right_on='id'
) \
.query('type == "Semantics"') \
.value_counts(subset=['id', 'label'])

Series([], Name: count, dtype: int64)

#### Pathways In-Degree

In [41]:
pd.merge(
    left=luma_edges_df,
    right=luma_nodes_df,
    how='inner',
    left_on='target',
    right_on='id'
) \
.query('type == "Pathway"') \
.value_counts(subset=['id', 'label'])

id             label                                      
R-HSA-162582   Signaling Pathways                             72
R-HSA-1266738  Developmental Biology                          40
R-HSA-9675108  Nervous system development                     21
R-HSA-9006934  Signaling by Receptor Tyrosine Kinases         18
R-HSA-1474244  Extracellular matrix organization               2
R-HSA-1474290  Collagen formation                              2
R-HSA-1650814  Collagen biosynthesis and modifying enzymes     2
R-HSA-383280   Nuclear Receptor transcription pathway          2
Name: count, dtype: int64

## Luminal B

In [42]:
group = 'luminal-b'

lumb_edges_df, lumb_nodes_df = create_lqn_network(group)

base_data_path = '../data/processed/networks/'
group_file_path = f'{base_data_path}/{group}'

edges_file_path = f'{group_file_path}/lqn-edges.csv'
lumb_edges_df.to_csv(edges_file_path, index=False)

nodes_file_path = f'{group_file_path}/lqn-nodes.csv'
lumb_nodes_df.to_csv(nodes_file_path, index=False)

In [43]:
# lumb_edges_df.to_csv('luminal-b_lqn-edges.csv', index=False)
# lumb_nodes_df.to_csv('luminal-b_lqn-nodes.csv', index=False)

In [44]:
lumb_edges_df

,source,target
0,hsa-miR-30d-5p,FST
1,hsa-miR-27b-3p,NEDD4
2,hsa-miR-381-3p,ESR1
3,hsa-miR-200c-3p,DZIP1
4,hsa-miR-342-3p,PDGFRA
...,...,...
491,S05932,R-HSA-5663202
492,S05933,R-HSA-5663202
493,TGFBR2,S05931
494,TGFBR2,S05932


In [45]:
lumb_nodes_df

,id,label,type
0,hsa-miR-200c-3p,hsa-miR-200c-3p,MicroRNA
1,hsa-miR-30d-5p,hsa-miR-30d-5p,MicroRNA
2,hsa-miR-27b-3p,hsa-miR-27b-3p,MicroRNA
3,hsa-miR-381-3p,hsa-miR-381-3p,MicroRNA
4,hsa-miR-33a-5p,hsa-miR-33a-5p,MicroRNA
...,...,...,...
292,S08411,Interferon Signaling > Antiviral mechanism by ...,Pathway Location
293,R-HSA-5663202,Diseases of signal transduction by growth fact...,Pathway
294,S05931,Signaling by TGF-beta Receptor Complex in Canc...,Pathway Location
295,S05932,Signaling by TGF-beta Receptor Complex in Canc...,Pathway Location


### Exploratory Analysis

#### Messenger RNAs Out-Degree

In [46]:
pd.merge(
    left=lumb_edges_df,
    right=lumb_nodes_df,
    how='inner',
    left_on='source',
    right_on='id'
) \
.query('type == "Messenger RNA"') \
.value_counts(subset=['id', 'label'])

id       label  
ESR1     ESR1       87
GNB5     GNB5       51
TGFBR2   TGFBR2     32
CSNK1A1  CSNK1A1    21
ARIH1    ARIH1      10
PDGFRA   PDGFRA      7
RPS6KA2  RPS6KA2     4
ABI2     ABI2        4
NF1      NF1         3
RASGRF2  RASGRF2     3
CDH6     CDH6        2
CHD3     CHD3        2
AXL      AXL         2
FST      FST         2
MMP2     MMP2        2
RND3     RND3        2
SOX17    SOX17       2
DZIP1    DZIP1       1
NEDD4    NEDD4       1
Name: count, dtype: int64

#### Semantics In-Degree

In [47]:
pd.merge(
    left=lumb_edges_df,
    right=lumb_nodes_df,
    how='inner',
    left_on='target',
    right_on='id'
) \
.query('type == "Semantics"') \
.value_counts(subset=['id', 'label'])

Series([], Name: count, dtype: int64)

In [48]:
lumb_edges_df \
    .query(
        '(target == "S01647")'
        + '| (target == "S01650")'
        + '| (target == "S01651")'
    ) \
    .sort_values(by='target')

,source,target
278,ESR1,S01647
279,GNB5,S01647
282,ESR1,S01650
283,GNB5,S01650
284,ESR1,S01651
285,GNB5,S01651


In [49]:
lumb_edges_df \
    .query(
        '(source == "S01647")'
        + '| (source == "S01650")'
        + '| (source == "S01651")'
    ) \
    .sort_values(by='source')

,source,target
68,S01647,R-HSA-162582
71,S01650,R-HSA-162582
72,S01651,R-HSA-162582


#### Semantics Out-Degree

In [50]:
pd.merge(
    left=lumb_edges_df,
    right=lumb_nodes_df,
    how='inner',
    left_on='source',
    right_on='id'
) \
.query('type == "Semantics"') \
.value_counts(subset=['id', 'label'])

Series([], Name: count, dtype: int64)

#### Pathways In-Degree

In [51]:
pd.merge(
    left=lumb_edges_df,
    right=lumb_nodes_df,
    how='inner',
    left_on='target',
    right_on='id'
) \
.query('type == "Pathway"') \
.value_counts(subset=['id', 'label'])

id             label                                                                           
R-HSA-162582   Signaling Pathways                                                                  210
R-HSA-1280215  Cytokine Signaling in Immune system                                                  12
R-HSA-9006934  Signaling by Receptor Tyrosine Kinases                                                7
R-HSA-5663202  Diseases of signal transduction by growth factor receptors and second messengers      3
R-HSA-418990   Adherens junctions interactions                                                       2
R-HSA-9006936  Signaling by TGFB family members                                                      1
Name: count, dtype: int64